# 日 K + 周 K：生图、训练、画结果

Run All 按顺序做完：画已走完的周 K，和日 K 配对训练，再画每天调仓的回测。参数只改下一格代码。

日 K 读 `k_pictures_gray_body`，不重画。周 K 已在 `k_pictures_gray_fuse_week`，这一版也不重画。权重和预测写到 `k_pictures_gray_fuse_day20`。日塔间隔 20 个交易日，周塔间隔 100，并且周塔的日子在日塔验证之后。见 `残差怎么训.md`。

In [ ]:
import sys
from pathlib import Path

import matplotlib
from matplotlib import font_manager as fm

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "change_rgb_i20"]
candidates.extend(cwd.parents)
for parent in [cwd, *cwd.parents]:
    candidates.append(parent / "change_rgb_i20")
seen = set()
root = None
for p in candidates:
    key = str(p)
    if key in seen:
        continue
    seen.add(key)
    if (p / "helpers.py").exists() and (p / "config.yaml").exists() and (p / "train_lib.py").exists():
        text = (p / "config.yaml").read_text(encoding="utf-8")
        if "task: regression" in text and (p / "images_lib.py").exists():
            root = p
            break
if root is None:
    raise FileNotFoundError(
        f"找不到 change_rgb_i20（config.yaml 里要有 task: regression）。cwd={cwd}"
    )
missing = [
    name for name in ("cnn.py", "train_lib.py", "backtest_lib.py", "images_lib.py", "draw_gray.py")
    if not (root / name).exists()
]
if missing:
    raise FileNotFoundError(f"{root} 缺少 {missing}。这些文件和 helpers.py 放在同一目录。")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import torch
from helpers import notebook_bootstrap
from images_lib import generate_images, verify_images
from train_lib import (
    plot_random_groups, plot_top10_excess, plot_top10_gross_net,
    predict_window, resolve_results_dir, train_window,
)
from backtest_lib import plot_decile_dashboard

font_candidates = [
    "Noto Sans CJK SC", "Noto Sans CJK JP",
    "WenQuanYi Zen Hei", "WenQuanYi Micro Hei",
    "Source Han Sans SC", "Source Han Sans CN",
    "Droid Sans Fallback", "SimHei", "Microsoft YaHei", "Arial Unicode MS",
]
available = {f.name for f in fm.fontManager.ttflist}
chosen = next((n for n in font_candidates if n in available), None)
if chosen is None:
    for font_path in Path("/usr/share/fonts").rglob("*"):
        if font_path.suffix.lower() in {".ttf", ".ttc", ".otf"} and any(
            k in font_path.name.lower() for k in ["noto", "wqy", "sourcehan", "cjk", "hei", "droid"]
        ):
            fm.fontManager.addfont(str(font_path))
            chosen = fm.FontProperties(fname=str(font_path)).get_name()
            break
if chosen is None:
    print("这台机器没找到中文字体，标题先用英文")
else:
    matplotlib.rcParams["font.sans-serif"] = [chosen, "DejaVu Sans"]
    matplotlib.rcParams["font.family"] = "sans-serif"
    matplotlib.rcParams["axes.unicode_minus"] = False
    print("使用字体:", chosen)

ROOT, cfg, paths = notebook_bootstrap()
print("ROOT   =", ROOT)
print("task   =", cfg.cnn.task, "epochs =", cfg.cnn.max_epochs, "lr =", cfg.cnn.optimizer.lr)
print("day    =", cfg.paths.images_remote)
print("week   =", cfg.paths.week_images_remote)
print("output =", cfg.paths.output_remote)
print("inputs =", list(cfg.fuse.inputs))
print("mode   =", cfg.fuse.mode)
print("cuda   =", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 参数

`WINDOW_KEY` 用 `"I20"` 或 `"I60"`。`MAX_CODES = None` 才是全量，图写到 144。改成 `20` 只抽样看拼图，后面的训练会停住，避免用 20 只股票训出结果。`JOBS` 是画图进程数。`N_SEEDS = 1` 先跑通。

In [ ]:
WINDOW_KEY = "I20"
MAX_CODES = None
DEBUG_CODES = None
COUNT_ONLY = False
JOBS = 20
USE_REMOTE = "auto"
N_SEEDS = 1
SKIP_EXISTING = True
ALLOW_SMALL_TRAIN = False

## 生成已走完的周 K

每个已走完的周一张图。周一到周四复用最近这一张，本周最后一个交易日才换成这一周。

In [ ]:
import importlib
import inspect
import images_lib

importlib.reload(images_lib)
generate_images = images_lib.generate_images
verify_images = images_lib.verify_images
if "window_key" not in inspect.signature(generate_images).parameters:
    raise RuntimeError(
        f"加载到的是旧 images_lib：{images_lib.__file__}\n"
        "用这一版 change_rgb_i20/images_lib.py 覆盖服务器上的文件，然后再跑。"
    )

info = generate_images(
    cfg, paths, window_key=WINDOW_KEY,
    max_codes=MAX_CODES,
    debug_codes=DEBUG_CODES,
    count_only=COUNT_ONLY,
    jobs=JOBS,
    require_volume=True,
    use_remote=USE_REMOTE,
)
print(info)

## 校验

拼图在 `figures/verify_{WINDOW_KEY}.png`。日期应该都是该周最后一个交易日。

In [ ]:
if COUNT_ONLY:
    raise RuntimeError("COUNT_ONLY=True 只统计张数，没有写出图片。全量跑把它改成 False。")
verify_images(cfg, paths, WINDOW_KEY, img_dir=info.get("img_dir"))

## 训练

两个卷积塔，权重不共享，特征拼接后一个输出。固定 10 个 epoch，按验证集 Spearman 留模型。已有的 `seed_k.pt` 会跳过。

In [ ]:
if MAX_CODES is not None and not ALLOW_SMALL_TRAIN:
    raise RuntimeError(
        f"MAX_CODES={MAX_CODES} 只画了部分股票。全量训练把 MAX_CODES 改成 None。"
        "确定只用这几只试跑时，把 ALLOW_SMALL_TRAIN 改成 True。"
    )

results = train_window(
    cfg, paths, WINDOW_KEY,
    n_seeds=N_SEEDS,
    num_workers=0,
    skip_existing=SKIP_EXISTING,
    pin_memory=False,
)
print(results)

## 样本外预测

`pred` 是预测的 5 日超额。多个种子先各自还原成超额，再取平均。

In [ ]:
pred = predict_window(cfg, paths, WINDOW_KEY, num_workers=0)
display(pred.head())

## 回测图

每个交易日按 `pred` 取最高 10%，持有 5 个交易日。因为每天都交易，5 个组合叠在一起。不满 5 个组合的日期不计入。预测文件在 `k_pictures_gray_fuse_day20/results/`。

In [ ]:
print("results =", resolve_results_dir(paths, cfg))
print("n_sleeves =", cfg.backtest.n_sleeves)
plot_top10_excess(cfg, paths, WINDOW_KEY)

In [ ]:
plot_decile_dashboard(cfg, paths, WINDOW_KEY)

In [ ]:
plot_random_groups(cfg, paths, window_key=WINDOW_KEY, n_groups=10)

In [ ]:
plot_top10_gross_net(cfg, paths, WINDOW_KEY, cost_bps_per_side=5)